# Automate FinOKF question vaults

Submit every subquestion in `questions.json` through the same local endpoints used by the UI. One multi-turn vault is created per question set and provider so the complete answer and evidence graph remain intact.

## Sources and behavior

- Questions: project `questions.json` (with `question.json` accepted as a fallback).
- Credentials: project `.env`; keys are sent with requests and are not written to vaults.
- Ollama: `http://127.0.0.1:11434`.
- Vaults: existing `POST /api/vaults` and `POST /api/chat` UI endpoints.
- Metrics: successful FinOKF and Naive results are upserted into `paper/question_metrics.csv`.

Start the FinOKF UI server before running the execution cell. No financial data is synthesized or replaced.

## Imports and paths

The server module supplies the established model defaults, company resolver, and browser index. Vault mutations still go through the UI API.

In [9]:
import csv
import html
import json
import os
import re
import sys
from pathlib import Path
from urllib import error as urlerror
from urllib import request as urlrequest
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

import serve_vault as server
from sec2md_processor import load_dotenv

## Configuration

`SKIP_EXISTING=True` avoids repeating expensive model calls when a vault with the expected name is already complete.

In [ ]:
BASE_URL = "http://127.0.0.1:8770"
QUESTIONS_PATH = next(
    (path for path in (ROOT / "questions.json", ROOT / "question.json") if path.is_file()),
    ROOT / "questions.json",
)
METRICS_PATH = ROOT / "paper" / "question_metrics.csv"
SKIP_EXISTING = True
INCLUDE_FAILED_METRICS = False

load_dotenv(ROOT / ".env")
PROVIDERS = [
    {"provider": "openai", "model": server.default_llm_model("openai"),
     "api_key": os.environ.get("OPENAI_API_KEY", "")},
    {"provider": "anthropic", "model": server.default_llm_model("anthropic"),
     "api_key": os.environ.get("ANTHROPIC_API_KEY", "")},
    # {"provider": "ollama", "model": server.default_llm_model("ollama"),
    #  "url": "http://127.0.0.1:11434"},
]

## API and input validation

Check the question schema, credentials, UI server, and all provider/model pairs before creating research vaults.

In [11]:
def request_json(method: str, path: str, payload: dict | None = None, timeout: int = 900) -> dict:
    body = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"} if body is not None else {}
    req = urlrequest.Request(BASE_URL + path, data=body, headers=headers, method=method)
    try:
        with urlrequest.urlopen(req, timeout=timeout) as response:
            result = json.loads(response.read() or b"{}")
    except urlerror.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:1000]
        raise RuntimeError(f"{method} {path} failed ({exc.code}): {detail}") from exc
    if result.get("ok") is False:
        raise RuntimeError(result.get("error") or f"{method} {path} failed")
    return result

question_document = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
question_sets = question_document.get("questions") or []
if not question_sets or any(not item.get("subquestions") for item in question_sets):
    raise ValueError(f"No complete question sets found in {QUESTIONS_PATH}")
for config in PROVIDERS:
    if config.get("provider") in {"openai", "anthropic"} and not config.get("api_key"):
        raise RuntimeError(f"Missing {config['provider']} API key in {ROOT / '.env'}")

request_json("GET", "/api/vaults", timeout=20)
provider_checks = [
    request_json("POST", "/api/verify-provider", {"provider_config": config})
    for config in PROVIDERS
]
provider_checks

[{'ok': True,
  'provider': 'ollama',
  'model': 'llama3.2:3b',
  'elapsed_ms': 295.947,
  'model_ms': 295.885,
  'usage': {'prompt_tokens': 44,
   'completion_tokens': 2,
   'total_tokens': 46,
   'usage_complete': True,
   'ollama_total_ms': 294.317,
   'ollama_load_ms': 2.341,
   'request_ids': []}}]

## Company routing and vault names

The first subquestion determines the starting company node, as it would in the UI. Later comparison questions stay in the same vault, allowing the server to add newly mentioned companies to the cumulative graph.

In [12]:
_browser_index, NODE_BY_ID = server.load_browser_index()

def company_node_for(question_set: dict) -> dict:
    prompt = question_set["subquestions"][0]["prompt"]
    tickers = server.mentioned_company_tickers(prompt, {}, NODE_BY_ID)
    if not tickers:
        raise ValueError(f"Could not resolve a company for {question_set['id']}")
    node = next((item for item in NODE_BY_ID.values()
                 if item.get("type") == "finance.entity" and item.get("ticker") == tickers[0]), None)
    if node is None:
        raise ValueError(f"No finance.entity node found for {tickers[0]}")
    return {key: node.get(key, "") for key in
            ("id", "title", "type", "ticker", "path", "finokf")}

def vault_title(question_set: dict, config: dict) -> str:
    model = re.sub(r"[^A-Za-z0-9._-]+", "-", config["model"]).strip("-")
    return f"{question_set['id']}-{model}"

[(item["id"], company_node_for(item)["ticker"]) for item in question_sets]

[('question-01', 'MSFT'), ('question-02', 'AAPL')]

## Metrics and graph checks

Metrics come from the persisted UI response. Existing CSV rows are preserved and matching rows are atomically updated after every successful turn.

In [13]:
METRIC_FIELDS = ["question_id", "agent", "question", "total_tokens", "elapsed_ms",
                 "provider", "model", "status", "source_vault", "source_turn"]

def compact_question_id(question_id: str) -> str:
    match = re.fullmatch(r"question-(\d+)-(\d+)", question_id)
    return f"{int(match.group(1))}.{int(match.group(2))}" if match else question_id

def metric_rows(question: dict, answers: list[dict], vault: dict, turn: int) -> list[dict]:
    rows = []
    for answer in answers:
        status = "ok" if answer.get("ok", True) else "failed"
        if status == "failed" and not INCLUDE_FAILED_METRICS:
            continue
        metrics = answer.get("metrics") or {}
        rows.append({
            "question_id": compact_question_id(question["id"]),
            "agent": answer.get("agent", ""), "question": question["prompt"],
            "total_tokens": metrics.get("total_tokens", ""),
            "elapsed_ms": metrics.get("total_ms", ""),
            "provider": answer.get("provider", ""), "model": answer.get("model", ""),
            "status": status, "source_vault": vault.get("slug", ""), "source_turn": turn,
        })
    return rows

def upsert_metrics(new_rows: list[dict]) -> None:
    METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    existing = []
    if METRICS_PATH.is_file():
        with METRICS_PATH.open(newline="", encoding="utf-8") as handle:
            existing = list(csv.DictReader(handle))
    keys = ("question_id", "agent", "provider", "model", "source_vault", "source_turn")
    indexed = {tuple(str(row.get(key, "")) for key in keys): row for row in existing}
    for row in new_rows:
        indexed[tuple(str(row.get(key, "")) for key in keys)] = row
    temporary = METRICS_PATH.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
        writer.writeheader()
        writer.writerows(indexed.values())
    temporary.replace(METRICS_PATH)

def validate_graph(vault: dict, expected_turns: int) -> None:
    graph_path = ROOT / vault["graph_path"]
    graph = json.loads(graph_path.read_text(encoding="utf-8"))
    if len(vault.get("runs") or []) != expected_turns:
        raise RuntimeError(f"{vault['title']} saved an unexpected number of turns")
    if not isinstance(graph.get("nodes"), list) or not isinstance(graph.get("links"), list):
        raise RuntimeError(f"{graph_path} is not a valid FinOKF graph")
    if expected_turns and not graph["nodes"]:
        raise RuntimeError(f"{graph_path} lost its answer graph")

## Naive and FinOKF comparison

Each completed question displays both full answers followed by the same metadata fields used by the UI's **Compare metadata** panel. Naive is shown first.

In [14]:
def ui_number(value) -> str:
    if value is None:
        return "Unavailable"
    return f"{value:,}" if isinstance(value, (int, float)) else str(value)

def ui_metadata(result: dict) -> dict[str, str]:
    metrics = result.get("metrics") or {}
    usage = metrics.get("usage_complete")
    token_accounting = ("Incomplete: provider usage unavailable for some calls" if usage is False
                        else "Complete" if usage is True else "Not recorded")
    no_inference = result.get("route") == "compiled-program" and metrics.get("total_tokens") == 0
    return {
        "Status": "Answered" if result.get("ok") else "Failed",
        "Provider / model": f"{result.get('provider') or '—'} / {result.get('model') or '—'}",
        "Data access": result.get("data_access") or "—", "Route": result.get("route") or "—",
        "Experiment": result.get("experiment") or "standard",
        "Cache / calculation": result.get("cache_kind") or ("cache hit" if result.get("cache_hit") else "uncached"),
        "Cache hit": "Yes" if result.get("cache_hit") else "No",
        "Elapsed (ms)": ui_number(metrics.get("total_ms")), "Non-model elapsed (ms)": ui_number(metrics.get("local_ms")),
        "Model time (ms)": ui_number(metrics.get("model_ms")), "Model calls": ui_number(metrics.get("model_calls")),
        "Model attempts": ui_number(metrics.get("model_attempts")), "Token accounting": token_accounting,
        "Inference record": "Historical calculation: no model call" if no_inference else "Provider-reported usage",
        "Routing (ms)": ui_number(metrics.get("route_ms")), "Binding / arithmetic (ms)": ui_number(metrics.get("bind_ms")),
        "Vault persistence (ms)": ui_number(metrics.get("persist_ms")), "Input tokens": ui_number(metrics.get("prompt_tokens")),
        "Output tokens": ui_number(metrics.get("completion_tokens")), "Total tokens": ui_number(metrics.get("total_tokens")),
        "Request IDs": ", ".join(metrics.get("request_ids") or []) or "Not recorded",
        "Web searches": ui_number(metrics.get("web_requests")), "Web page requests": ui_number(metrics.get("page_requests")),
        "Web pages read": ui_number(metrics.get("pages_fetched")), "Evidence sources": ui_number(metrics.get("source_count")),
    }


In [15]:
def answers_for_turn(vault: dict, turn: int) -> list[dict]:
    results = [message["result"] for message in vault.get("messages") or []
               if message.get("turn") == turn and isinstance(message.get("result"), dict)]
    return sorted(results, key=lambda item: 0 if item.get("agent") == "naive" else 1)

def display_agent_comparison(question: dict, answers: list[dict]) -> None:
    by_agent = {answer.get("agent"): answer for answer in answers}
    display(Markdown(f"### {question['id']}\n\n> {question['prompt']}"))
    for agent, label, subtitle in (("naive", "Naive", "Independent web research · uncached"),
                                   ("finokf", "FinOKF", "Local evidence + web research · cache enabled")):
        result = by_agent.get(agent) or {"ok": False, "answer": "No result returned."}
        display(Markdown(f"#### {label}\n\n*{subtitle}*\n\n{result.get('answer') or 'No answer returned.'}"))
    naive = ui_metadata(by_agent.get("naive") or {})
    finokf = ui_metadata(by_agent.get("finokf") or {})
    rows = "".join(f"<tr><th style='text-align:left'>{html.escape(label)}</th>"
                   f"<td>{html.escape(naive[label])}</td><td>{html.escape(finokf[label])}</td></tr>"
                   for label in naive)
    display(HTML("<table><caption><strong>Measured for this question</strong></caption>"
                 "<thead><tr><th>Metadata</th><th>Naive</th><th>FinOKF</th></tr></thead>"
                 f"<tbody>{rows}</tbody></table>"))


## Submit one question set

Create the named vault, then send every subquestion as a turn using the same vault ID. The server runs both default agents, writes result files, and rebuilds the graph.

In [16]:
def run_question_set(question_set: dict, config: dict, known_vaults: dict[str, dict]) -> dict:
    title = vault_title(question_set, config)
    expected_turns = len(question_set["subquestions"])
    existing = known_vaults.get(title)
    if SKIP_EXISTING and existing and len(existing.get("runs") or []) == expected_turns:
        validate_graph(existing, expected_turns)
        for turn, question in enumerate(question_set["subquestions"], start=1):
            answers = answers_for_turn(existing, turn)
            display_agent_comparison(question, answers)
            upsert_metrics(metric_rows(question, answers, existing, turn))
        return {"title": title, "status": "skipped-complete", "vault": existing}
    if existing and existing.get("runs"):
        raise RuntimeError(f"{title} exists but is incomplete; resolve it before rerunning.")

    node = company_node_for(question_set)
    vault = request_json("POST", "/api/vaults", {"title": title, "node": node})["vault"]
    for turn, question in enumerate(question_set["subquestions"], start=1):
        result = request_json("POST", "/api/chat", {
            "message": question["prompt"], "stream": False, "provider_config": config,
            "vault_id": vault["vault_id"], "markdown": "", "node": node, "neighbors": [],
        })
        if result.get("persistence_error"):
            raise RuntimeError(result["persistence_error"])
        vault = result["vault"]
        validate_graph(vault, turn)
        answers = sorted(result.get("answers") or [], key=lambda item: 0 if item.get("agent") == "naive" else 1)
        display_agent_comparison(question, answers)
        upsert_metrics(metric_rows(question, answers, vault, turn))
        print(f"{title}: completed {question['id']} ({turn}/{expected_turns})")
    return {"title": title, "status": "completed", "vault": vault}


## Execution plan

Review the names before starting. With the current input this creates six vaults: question sets 01 and 02 for the OpenAI, Anthropic, and Ollama default models.

In [17]:
execution_plan = [
    {"question_set": item["id"], "provider": config["provider"],
     "model": config["model"], "vault": vault_title(item, config),
     "turns": len(item["subquestions"])}
    for item in question_sets for config in PROVIDERS
]
execution_plan

[{'question_set': 'question-01',
  'provider': 'ollama',
  'model': 'llama3.2:3b',
  'vault': 'question-01-llama3.2-3b',
  'turns': 3},
 {'question_set': 'question-02',
  'provider': 'ollama',
  'model': 'llama3.2:3b',
  'vault': 'question-02-llama3.2-3b',
  'turns': 4}]

## Run all providers

This is the long-running cell. Providers run sequentially so activity stays attributable; progress prints after every persisted turn.

In [18]:
known_vaults = {item.get("title"): item for item in request_json("GET", "/api/vaults")["vaults"]}
run_summary = []
for question_set in question_sets:
    for config in PROVIDERS:
        outcome = run_question_set(question_set, config, known_vaults)
        vault = outcome["vault"]
        run_summary.append({
            "vault": outcome["title"], "provider": config["provider"],
            "model": config["model"], "status": outcome["status"],
            "turns": len(vault.get("runs") or []), "graph_path": vault.get("graph_path"),
        })
        known_vaults[outcome["title"]] = vault

### question-01-01

> How did the profitability of Microsoft's growth change across FY2023, FY2024 and FY2025? Assess consolidated operating leverage and whether the profitability of additional revenue strengthened or weakened.

#### Naive

*Independent web research · uncached*

**Assessment of Microsoft's Profitability across FY2023, FY2024, and FY2025**

Given the limitations of the provided web evidence, a comprehensive analysis of Microsoft's profitability across the specified years is not feasible. The available information includes:

1.  Microsoft's 2025 Annual Report, which provides an overview of the company's performance for the fiscal year ending June 30, 2025.
2.  The 2024 Annual Report and 2023 Annual Report, which offer insights into the company's performance for the previous fiscal years.
3.  Various search results, including news articles and SEC filings, which provide additional context and information about Microsoft's business operations and financial performance.

However, the provided web evidence does not include the revenue figures for FY2023, FY2024, and FY2025. As a result, it is not possible to assess the profitability of Microsoft's growth across these periods.

**Consolidated Operating Leverage**

The provided web evidence does not include information on consolidated operating leverage. However, the 2025 Annual Report provides an overview of Microsoft's segments, including the Productivity and Business Processes segment, which includes Microsoft 365 Commercial cloud revenue growth.

**Conclusion**

Due to the limitations of the provided web evidence, a comprehensive analysis of Microsoft's profitability across FY2023, FY2024, and FY2025 is not feasible. The available information includes an overview of the company's performance for the fiscal year ending June 30, 2025, as well as various search results and SEC filings. However, the revenue figures for FY2023, FY2024, and FY2025 are not available, making it impossible to assess the profitability of Microsoft's growth across these periods.

**Table: Microsoft's Revenue Growth**

| Year | Revenue Growth |
| --- | --- |
| FY2023 | Not available |
| FY2024 | Not available |
| FY2025 | Not available |

**Sources:**

*   Microsoft 2025 Annual Report: <https://www.microsoft.com/investor/reports/ar25/index.html>
*   Microsoft 2024 Annual Report: <https://www.microsoft.com/en-us/investor/annual-reports>
*   Microsoft 2023 Annual Report: <https://www.microsoft.com/en-us/investor/annual-reports>
*   Various search results: <https://www.google.com/search>
*   SEC Filings: <https://www.sec.gov/Archives/edgar/data/789019/000095017025100235/0000950170-25-100235-index.htm>

#### FinOKF

*Local evidence + web research · cache enabled*

**Conclusion:** Microsoft's consolidated operating margin increased by 287.1 basis points from FY2023 to FY2024 and by 97.8 basis points from FY2024 to FY2025, indicating strengthened profitability of additional revenue.

**Drivers:** The increase in operating margin can be attributed to the growth in revenue, which led to a more efficient use of resources and better cost management. The incremental operating margin, which measures the increase in operating margin relative to revenue growth, indicates that Microsoft's profitability improved at a faster rate than its revenue growth.

**Quantifications:**

| Fiscal year (exact period) | Revenue (USD millions) | Operating Income (USD millions) | Operating margin |
| --- | ---: | ---: | ---: |
| FY2023 | 211,915.000 | 88,523.000 | 41.77% |
| FY2024 | 245,122.000 | 109,433.000 | 44.64% |
| FY2025 | 281,724.000 | 128,528.000 | 45.62% |

**Comparison:**

| Fiscal year (exact period) | Revenue Growth | Operating Income Growth | Incremental Operating Margin |
| --- | ---: | ---: | ---: |
| FY2023 → FY2024 | 33,207.000 million | 20,910.000 million | 62.97% |
| FY2024 → FY2025 | 36,602.000 million | 19,095.000 million | 52.17% |

**Limitations:** The analysis is based on reported-period calculations and does not establish the cause or future persistence of the change in operating margin. Additionally, the incremental operating margin does not account for potential changes in operating expenses or capital expenditures.

**Sources:**

* `filings/MSFT/MSFT-FY2025-10-K-2025-07-30-0000950170-25-100235.md`
* `data/prices/sp100_yahoo/daily` for historical stock prices
* Local CSV evidence for revenue, operating income, and operating margin data.

Metadata,Naive,FinOKF
Status,Answered,Answered
Provider / model,ollama / llama3.2:3b,ollama / llama3.2:3b
Data access,Public web search and page retrieval; independent numerical review; uncached,Local filings and stock prices from data/prices; web research for missing non-price evidence
Route,web-research,ollama-measured-synthesis
Experiment,standard baseline,standard
Cache / calculation,uncached,calculation-lru
Cache hit,No,Yes
Elapsed (ms),"49,603.4","6,516.364"
Non-model elapsed (ms),Unavailable,82.344
Model time (ms),"30,246.565","6,434.02"


question-01-llama3.2-3b: completed question-01-01 (1/3)


...

### question-02-04

> How did MSFT's consolidated operating leverage change between FY2024 and FY2025 as compared to Apple?

#### Naive

*Independent web research · uncached*

**Research Review and Analysis**

The research review was not structured, and an additional financial-statement search was performed to obtain the necessary information.

**Gaps in the Data**

The operating leverage figures for MSFT and Apple for FY2024 and FY2025 are not available.

**Available Data**

The available data includes:

* MSFT's 2025 Annual Report, which provides revenue and gross margin information for FY2025.
* MSFT's 2024 Annual Report, which provides revenue and gross margin information for FY2024.
* Accilize's financial analysis report, which provides a summary of MSFT's financial performance for FY2024.
* Apple's FY2024 Q4 financial statements, which provide revenue and gross margin information for FY2024.

**Analysis**

Based on the available data, it is not possible to calculate MSFT's consolidated operating leverage for FY2024 and FY2025. However, we can calculate Apple's consolidated operating leverage for FY2024 using the available data.

**Calculation**

To calculate Apple's consolidated operating leverage for FY2024, we need to calculate the gross margin and operating income for FY2024.

Gross Margin:
Apple's FY2024 Q4 financial statements provide the following gross margin information:

* Gross margin: $42.271 billion
* Gross margin percentage: 44.7%

Operating Income:
Apple's FY2024 Q4 financial statements provide the following operating income information:

* Operating income: $42.271 billion
* Operating income percentage: 44.7%

Operating Leverage:
Operating leverage can be calculated using the following formula:

Operating Leverage = (Gross Margin / Revenue) x 100

Using the available data, we can calculate Apple's consolidated operating leverage for FY2024 as follows:

* Revenue: $94.9 billion (FY2024 Q4 financial statements)
* Gross Margin: $42.271 billion (FY2024 Q4 financial statements)
* Operating Leverage: (42.271 / 94.9) x 100 = 44.5%

**Conclusion**

Based on the available data, Apple's consolidated operating leverage for FY2024 is 44.5%. However, MSFT's consolidated operating leverage for FY2024 and FY2025 are not available.

**Compact Table**

| Company | FY2024 | FY2025 |
| --- | --- | --- |
| Apple | 44.5% |  |
| MSFT |  |  |

**Sources**

* Apple's FY2024 Q4 financial statements
* Accilize's financial analysis report
* MSFT's 2025 Annual Report
* MSFT's 2024 Annual Report

#### FinOKF

*Local evidence + web research · cache enabled*

**Conclusion:** MSFT's consolidated operating leverage increased by 97.8 basis points between FY2024 and FY2025, while AAPL's operating leverage increased by 46.1 basis points during the same period.

**Drivers:** The increase in operating leverage for both companies can be attributed to their ability to maintain or increase profitability despite revenue growth. MSFT's operating margin expanded from 44.64% to 45.62%, while AAPL's operating margin increased from 31.51% to 31.97%. These improvements in profitability can be attributed to various factors, including cost management, pricing strategies, and operational efficiencies.

**Quantitative Comparison:**

| Company | FY2024 | FY2025 | ΔRevenue | ΔOperating Income | ΔOperating Margin |
| --- | ---: | ---: | ---: | ---: | ---: |
| MSFT | 44.64% | 45.62% | +97.8 basis points |
| AAPL | 31.51% | 31.97% | +46.1 basis points |

**Calculations:**

* ΔRevenue = Revenue FY2025 - Revenue FY2024
* ΔOperating Income = Operating Income FY2025 - Operating Income FY2024
* ΔOperating Margin = (Operating Income FY2025 - Operating Income FY2024) / (Revenue FY2025 - Revenue FY2024) × 100

**Limitations:** This analysis is based on reported-period calculations and does not establish the cause or future persistence of the changes in operating leverage. Additional factors, such as changes in cost structures, pricing strategies, and market conditions, may have contributed to these changes.

**Sources:**

* MSFT: `filings/MSFT/MSFT-FY2025-10-K-2025-07-30-0000950170-25-100235.md`
* AAPL: `filings/AAPL/AAPL-FY2025-10-K-2025-10-31-0000320193-25-000079.md`

Metadata,Naive,FinOKF
Status,Answered,Answered
Provider / model,ollama / llama3.2:3b,ollama / llama3.2:3b
Data access,Public web search and page retrieval; independent numerical review; uncached,Local filings and stock prices from data/prices; web research for missing non-price evidence
Route,web-research,ollama-measured-synthesis
Experiment,standard baseline,standard
Cache / calculation,uncached,measured-evidence
Cache hit,No,No
Elapsed (ms),"59,149.97","7,819.216"
Non-model elapsed (ms),Unavailable,139.254
Model time (ms),"38,245.799","7,679.962"


question-02-llama3.2-3b: completed question-02-04 (4/4)


## Results

Confirm every vault has its expected turns and graph path, then refresh Home in the FinOKF UI.

In [19]:
run_summary

[{'vault': 'question-01-llama3.2-3b',
  'provider': 'ollama',
  'model': 'llama3.2:3b',
  'status': 'completed',
  'turns': 3,
  'graph_path': 'data/vaults/answers/msft-20260908t210244795523z-ee17c56a735d/graph.json'},
 {'vault': 'question-02-llama3.2-3b',
  'provider': 'ollama',
  'model': 'llama3.2:3b',
  'status': 'completed',
  'turns': 4,
  'graph_path': 'data/vaults/answers/aapl-20260908t210510856010z-7a6e05bed7e8/graph.json'}]